## Planning:
- Load data
- Create Feature Engineering + No Dummify
- Model XGBoost, standard parameter, 5 cv

In [2]:
# data processing
import pandas as pd
import numpy as np

# data visualisation
import seaborn as sns
import matplotlib.pyplot as plt

# data modeling 
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# model 
from sklearn.linear_model import LogisticRegression
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier, Pool

In [3]:
import gc

In [26]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
submit = pd.read_csv("sample_submission.csv")

In [27]:
train['Arrival Delay in Minutes'] = train['Arrival Delay in Minutes'].fillna(train['Departure Delay in Minutes'])

In [28]:
test['Arrival Delay in Minutes'] = test['Arrival Delay in Minutes'].fillna(test['Departure Delay in Minutes'])

In [29]:
# check correlation of numeric variable
# take all column numeric column
numeric_cols = train.drop(columns='id').select_dtypes(include='number').columns.tolist()

In [30]:
non_numeric_cols = train.select_dtypes(exclude='number').columns.tolist()

## Feature Engineering Mumu

In [62]:
# splitting X and X_test, take out satisfaction
X = train.drop(columns=["id","satisfaction"])
y = train["satisfaction"]
X_test = test.drop(columns=["id"])

Inspiration: https://www.kaggle.com/competitions/playground-series-s6e10/discussion/754317

In [63]:
svc = ['Inflight wifi service','Departure/Arrival time convenient','Ease of Online booking',
       'Gate location','Food and drink','Online boarding','Seat comfort','Inflight entertainment',
       'On-board service','Leg room service','Baggage handling','Checkin service','Cleanliness']

In [64]:
# menghitung rata-rata rating dan total rating pada setiap row
for df in (X, X_test):
    df['svc_sum'] = df[svc].sum(axis=1)
    df['svc_mean'] = df[svc].mean(axis=1)

In [65]:
X.head(2)

,Age,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,Gate location,Food and drink,Online boarding,Seat comfort,Inflight entertainment,...,Online boarding_FE,Seat comfort_FE,Inflight entertainment_FE,On-board service_FE,Leg room service_FE,Baggage handling_FE,Checkin service_FE,Cleanliness_FE,svc_sum,svc_mean
0,36,694,4,4,4,3,1,4,1,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,37,2.846154
1,56,651,5,5,5,5,5,4,4,5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,60,4.615385


In [66]:
# route stats: train + test pooled, pure feature statistics (no target anywhere)
tmp = pd.concat([X[['Flight Distance','Age','svc_sum']+svc],
                 X_test[['Flight Distance','Age','svc_sum']+svc]], ignore_index=True)
g = tmp.groupby('Flight Distance')

In [67]:
route = g.size().to_frame('route_count')

In [68]:
# in each of the variable, create new variable "mean" based on each group
route['route_age_mean'] = g['Age'].mean()
route['route_svcsum_mean'] = g['svc_sum'].mean()

In [69]:
# did the same thing as above with more automatic way
for c in svc:
    route['r_' + c[:12] + '_m'] = g[c].mean()

In [70]:
route = route.fillna(0)

X = X.join(route, on='Flight Distance')
X_test = X_test.join(route, on='Flight Distance')

## Feature Engineering Naji

In [71]:
CAT_FEATURES = ['Gender', 'Customer Type', 'Type of Travel', 'Class']
RATING_COLS = [
    'Inflight wifi service', 'Departure/Arrival time convenient', 'Ease of Online booking', 
    'Food and drink', 'Online boarding', 'Seat comfort', 'Inflight entertainment', 
    'On-board service', 'Leg room service', 'Baggage handling', 'Checkin service', 'Cleanliness'
]

In [72]:
# 🧮 Generating Domain Features & Pairs
for df in [X, X_test]:
    df['biz_elite_nj'] = ((df['Type of Travel'] == 'Business travel') & (df['Class'] == 'Business')).astype(np.int8)
    df['Wifi_Booking_pair'] = df['Inflight wifi service'].astype(str) + "_" + df['Ease of Online booking'].astype(str)
    df['Wifi_Time_pair']    = df['Inflight wifi service'].astype(str) + "_" + df['Departure/Arrival time convenient'].astype(str)
    df['Boarding_Wifi_pair']= df['Online boarding'].astype(str) + "_" + df['Inflight wifi service'].astype(str)
    df['Class_Travel_pair'] = df['Class'].astype(str) + "_" + df['Type of Travel'].astype(str)
    df['Cust_Travel_pair']  = df['Customer Type'].astype(str) + "_" + df['Type of Travel'].astype(str)
    df['Class_Cust_pair']   = df['Class'].astype(str) + "_" + df['Customer Type'].astype(str)
    df['Class_Wifi_pair']     = df['Class'].astype(str) + "_" + df['Inflight wifi service'].astype(str)
    df['Class_Boarding_pair'] = df['Class'].astype(str) + "_" + df['Online boarding'].astype(str)
    df['Travel_Wifi_pair']    = df['Type of Travel'].astype(str) + "_" + df['Inflight wifi service'].astype(str)
    df['Travel_Boarding_pair']= df['Type of Travel'].astype(str) + "_" + df['Online boarding'].astype(str)
    df['Seat_Ent_pair']       = df['Seat comfort'].astype(str) + "_" + df['Inflight entertainment'].astype(str)
    df['Route_Class_pair'] = df['Flight Distance'].astype(str) + "_" + df['Class'].astype(str)
    df['Route_Travel_pair'] = df['Flight Distance'].astype(str) + "_" + df['Type of Travel'].astype(str)

INTERACTION_COLS = [
    'Wifi_Booking_pair', 'Wifi_Time_pair', 'Boarding_Wifi_pair', 'Class_Travel_pair', 'Cust_Travel_pair', 
    'Class_Cust_pair', 'Class_Wifi_pair', 'Class_Boarding_pair', 'Travel_Wifi_pair', 'Travel_Boarding_pair', 
    'Seat_Ent_pair', 'Route_Class_pair', 'Route_Travel_pair'
]

# 🔢 Extracting  Digits
DIGIT_MAP = {'Age': [0, 1], 'Flight Distance': [0, 1, 2, 3], 'Departure Delay in Minutes': [1], 'Arrival Delay in Minutes': [1]}
digit_feats_nj = [] 
for c, powers in DIGIT_MAP.items():
    for k in powers:
        col_name = f"{c}_digit{k}"
        X[col_name] = (X[c].fillna(0) // (10**k) % 10).astype(np.int8)
        X_test[col_name] = (X_test[c].fillna(0) // (10**k) % 10).astype(np.int8)
        digit_feats_nj.append(col_name)

# ✈️ Extracting Route Profiles based on Flight Distance
for df in [X, X_test]:
    df['Route_ID'] = df['Flight Distance'].astype(int)

route_counts = pd.concat([X['Route_ID'], X_test['Route_ID']], ignore_index=True).value_counts()
for df in [X, X_test]:
    df['Route_Freq'] = df['Route_ID'].map(route_counts).astype(np.int32)


In [73]:
route_profile_cols = ['Age', 'Departure Delay in Minutes'] + RATING_COLS

In [74]:
route_profile = X.groupby('Route_ID')[route_profile_cols].mean()
route_profile.columns = [f"Route_Mean_{c.replace(' ', '_').replace('/', '_')}" for c in route_profile.columns]
route_profile = route_profile.reset_index()

In [75]:
X = X.merge(route_profile, on="Route_ID", how="left")
X_test  = X_test.merge(route_profile,  on="Route_ID", how="left")

In [76]:
for c in route_profile.columns:
    if c == "Route_ID": continue
    X[c] = X[c].fillna(X[c].mean())
    X_test[c]  = X_test[c].fillna(X[c].mean())

X.drop(columns=['Route_ID'], inplace=True)
X_test.drop(columns=['Route_ID'], inplace=True)

# Setup Encoding
TARGET_ENCODE_COLS = CAT_FEATURES + RATING_COLS + digit_feats_nj + INTERACTION_COLS

num_to_cat_cols = []
for c in ['Age', 'Flight Distance']: 
    cat_name = f"{c}_cat"
    X[cat_name] = X[c].fillna(-1).astype(str)
    X_test[cat_name] = X_test[c].fillna(-1).astype(str)
    num_to_cat_cols.append(cat_name)

TARGET_ENCODE_COLS.extend(num_to_cat_cols)

for df in [X, X_test]:
    for col in CAT_FEATURES + digit_feats_nj:
        df[col] = df[col].astype(str)
    for col in RATING_COLS:
        df[col] = "Rating_" + df[col].astype(str)

In [77]:
# 📊 Applying Frequency Encoding
combined_nj = pd.concat([X, X_test], ignore_index=True)
for col in TARGET_ENCODE_COLS:
    freq_mapping = combined_nj[col].value_counts(normalize=True).to_dict()
    X[f"{col}_FE"] = X[col].map(freq_mapping).astype(np.float32)
    X_test[f"{col}_FE"]  = X_test[col].map(freq_mapping).astype(np.float32)
del combined_nj; gc.collect()

# Hardcoded Pruning List
FEATURES_TO_DROP = [
    'Departure Delay in Minutes_digit1_FE', 'Departure Delay in Minutes_digit1_TE_auto',
    'Arrival Delay in Minutes_digit1_FE', 'Arrival Delay in Minutes_digit1_TE_auto',
    'Departure Delay in Minutes', 'Flight Distance_digit3_FE', 'Flight Distance_digit1_FE',
    'Age_digit0_FE', 'Flight Distance_digit0_FE', 'Flight Distance_digit2_FE','Age_digit0_FE'
]

## Basic Data Process for Modeling

Data preparation - Create Dummy variable - NO DUMMY

In [89]:
for col in ['Route_Class_pair', 'Route_Travel_pair']: # plus other categorical columns
    combined_categories = pd.concat([X[col], X_test[col]]).astype('category').cat.categories
    X[col] = pd.Categorical(X[col], categories=combined_categories)
    X_test[col] = pd.Categorical(X_test[col], categories=combined_categories)

In [90]:
for df in [X, X_test]:
    for col in df.select_dtypes(include=['object']).columns:
        df[col] = df[col].astype('category')

In [91]:
X.shape

(699635, 117)

In [92]:
X_test.shape

(299844, 117)

## Model using XGboost 5 fold CV

In [94]:
SEED = 67
N_FOLDS = 5
TARGET = 'satisfaction'

In [95]:
# Stratified KFold cross validation
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
FOLDS = list(skf.split(X, y))

In [96]:
def run_cv(fit_predict_fn, name):
    oof = np.zeros(len(X))
    test_pred = np.zeros(len(X_test))
    scores = []
    for fold, (tr_idx, va_idx) in enumerate(FOLDS):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]
        va_pred, te_pred = fit_predict_fn(X_tr, y_tr, X_va, y_va)
        oof[va_idx] = va_pred
        test_pred += te_pred / N_FOLDS
        auc = roc_auc_score(y_va, va_pred)
        scores.append(auc)
        print(f'  fold {fold}: AUC = {auc:.5f}')
    total = roc_auc_score(y, oof)
    print(f'{name} | OOF AUC = {total:.5f} (mean {np.mean(scores):.5f} ± {np.std(scores):.5f})')
    return oof, test_pred, total

In [97]:
final_params = {
    # fix parameter
        "objective":"binary:logistic",
        "eval_metric": "auc",
        "n_estimators": 5000,
        "tree_method": "hist",
        "enable_categorical": True,
        "random_state": 67,
        "colsample_bytree":0.8,
        "subsample":0.8,
        "n_jobs":-1,
        # early stopping round
        "early_stopping_rounds": 200,

        "learning_rate": 0.023621412607945612,
        "max_depth": 8,
        "reg_lambda": 0.15767822496360231,
        "alpha": 0.3574854835852679,
        "min_child_weight":29
}

In [98]:
def fit_xgb(X_tr, y_tr, X_va, y_va):
    model = xgb.XGBClassifier(**final_params)
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    return (model.predict_proba(X_va)[:, 1],
            model.predict_proba(X_test)[:, 1])

In [99]:
oof_xgb, test_xgb, auc_xgb = run_cv(fit_xgb, 'XGBoost')

  fold 0: AUC = 0.96013
  fold 1: AUC = 0.95983
  fold 2: AUC = 0.95969
  fold 3: AUC = 0.96017
  fold 4: AUC = 0.96017
XGBoost | OOF AUC = 0.96000 (mean 0.96000 ± 0.00020)


In [100]:
def save_sub(pred, name):
    sub = submit.copy()
    sub['satisfaction'] = pred
    sub.to_csv(name, index=False)
    print('saving file:', name)

In [101]:
save_sub(test_xgb,'sub_xgb_17_5cv_FE_mumu_naji.csv')

saving file: sub_xgb_17_5cv_FE_mumu_naji.csv


In [102]:
oof_xgb_df = train[['id']].copy()
oof_xgb_df.loc[:,'oof_xgb_17_5cv_fe_mumu_naji'] = oof_xgb
oof_xgb_df.to_csv('oof_xgb_17_5cv_fe_mumu_naji.csv')